In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import shap
from imblearn.over_sampling import SMOTE


/Users/garimau/dev/dsdp-ucsf/UCSFDelirium/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from collections import Counter

# Load data (handling potential byte order mark)
admission_annotation = pd.read_csv('/Users/garimau/dev/dsdp-ucsf/UCSFDelirium/AdmissionDemographicGenes/Raw Data Files/admission_annotation.csv')
gene_symbols = pd.read_csv('/Users/garimau/dev/dsdp-ucsf/UCSFDelirium/AdmissionDemographicGenes/Raw Data Files/gene_symbols.csv')
admission_norm_gene_exp_df = pd.read_csv('/Users/garimau/dev/dsdp-ucsf/UCSFDelirium/AdmissionDemographicGenes/Raw Data Files/admission_norm_gene_exp_df.csv', index_col=0)


# Merge and clean gene expression data
admission_norm_gene_exp_df = admission_norm_gene_exp_df.merge(gene_symbols, left_index=True, right_on='gene_ids', how='left')
admission_norm_gene_exp_df = admission_norm_gene_exp_df.set_index('gene_symbols').drop(['gene_ids', 'Unnamed: 0'], axis=1)
admission_norm_gene_exp_df = admission_norm_gene_exp_df.dropna(axis=0)

# Remove hyphens from 'X' column in admission_annotation
admission_annotation['X'] = admission_annotation['X'].str.replace('-', '')

# Transpose, merge, and prepare data
X = admission_norm_gene_exp_df.transpose()
X = X.merge(admission_annotation, left_index=True, right_on='X', how='inner').set_index('X')
X.to_csv('admission.csv', index=False)

In [5]:
y = X['Diagnosis']
#X = X.drop('Diagnosis', axis=1)  # Keep 'Steroids'
X = X.drop('Diagnosis', axis=1)
X.columns = X.columns.astype(str)

In [6]:
oversample = SMOTE()
counter = Counter(y)
print(counter)
X.head()

X, y = oversample.fit_resample(X, y)
# summarize the new class distribution
# scatter plot of examples by class label
X.shapecounter = Counter(y)
print(counter)

# Assuming X is a DataFrame and y is a Series or array
X_resampled = pd.DataFrame(X)
y_resampled = pd.Series(y, name='target')

# Concatenate X and y
df_resampled = pd.concat([X_resampled, y_resampled], axis=1)

# Save to CSV
df_resampled.to_csv('oversampled_admission.csv', index=False)


Counter({0: 88, 1: 24})


/var/folders/ql/sqk652rs4vdbmts6q6jhtcv00000gn/T/ipykernel_58898/1948636158.py:9: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  X.shapecounter = Counter(y)


Counter({0: 88, 1: 24})
